# Digit Count Classifier — Training Notebook

Trains a ResNet34-based binary classifier to predict whether a jersey crop shows **1 digit** (0–9) or **2 digits** (10–99).

**Requirements:** Colab GPU runtime (Runtime → Change runtime type → GPU)

In [ ]:
# Verify GPU is available
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi

In [ ]:
# Mount Google Drive (for saving trained model later)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo and checkout the MoviNet branch
%cd /content
!git clone https://github.com/magnimel/jersey-number-pipeline.git
%cd jersey-number-pipeline
!git checkout arjun/MoviNet

In [ ]:
# Install dependencies (do NOT install torch/torchvision — Colab already has CUDA versions)
!pip install -q SoccerNet gdown tqdm pillow

In [ ]:
# Download SoccerNet jersey-2023 data (cached on Drive to avoid re-downloading)
import os, zipfile, glob, shutil
from pathlib import Path

data_dir = Path("/content/jersey-number-pipeline/data")
soccer_dir = data_dir / "SoccerNet"
drive_cache = Path("/content/drive/MyDrive/jersey-number-pipeline-data/SoccerNet")

# Check if cached on Drive
if drive_cache.exists() and (drive_cache / "train" / "train_gt.json").exists():
    print("Loading cached data from Google Drive...")
    shutil.copytree(str(drive_cache), str(soccer_dir))
    print("Done! Loaded from Drive cache.")
else:
    print("No cache found — downloading fresh...")
    from SoccerNet.Downloader import SoccerNetDownloader as SNdl
    dl = SNdl(LocalDirectory=str(data_dir))
    dl.downloadDataTask(task="jersey-2023", split=["train", "test", "challenge"])

    # Find and unzip all zips
    all_zips = glob.glob(str(data_dir / "**" / "*.zip"), recursive=True)
    print(f"Found {len(all_zips)} zip files")
    for zp in all_zips:
        parent = os.path.dirname(zp)
        print(f"Extracting {os.path.basename(zp)}...")
        with zipfile.ZipFile(zp, 'r') as zf:
            zf.extractall(parent)

    # Find jersey-2023 folder and rename to SoccerNet
    jersey_dirs = glob.glob(str(data_dir / "**" / "jersey-2023"), recursive=True)
    if not jersey_dirs:
        jersey_dirs = [str(data_dir / "jersey-2023")]
    source = jersey_dirs[0]
    if os.path.exists(source) and str(source) != str(soccer_dir):
        if soccer_dir.exists():
            shutil.rmtree(soccer_dir)
        shutil.move(source, str(soccer_dir))

    # Cache to Drive for next time
    print("Caching data to Google Drive (one-time)...")
    drive_cache.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(str(soccer_dir), str(drive_cache))
    print("Cached!")

# Verify
for name, path in {
    "train_gt.json": soccer_dir / "train" / "train_gt.json",
    "test_gt.json": soccer_dir / "test" / "test_gt.json",
    "train/images": soccer_dir / "train" / "images",
}.items():
    print(f"{'OK  ' if path.exists() else 'MISS'} {name}")

In [ ]:
# Create val split (skip if already exists)
import json, random, shutil
from pathlib import Path

root = Path("/content/jersey-number-pipeline/data/SoccerNet")
val_gt_path = root / "val" / "val_gt.json"

if val_gt_path.exists():
    with open(val_gt_path) as f:
        val_gt = json.load(f)
    with open(root / "train" / "train_gt.json") as f:
        train_gt = json.load(f)
    print(f"Val split already exists — train: {len(train_gt)}, val: {len(val_gt)}")
else:
    seed = 42
    val_ratio = 0.15
    train_gt_path = root / "train" / "train_gt.json"
    train_img_dir = root / "train" / "images"
    val_img_dir = root / "val" / "images"

    random.seed(seed)
    val_img_dir.mkdir(parents=True, exist_ok=True)

    with open(train_gt_path, "r") as f:
        gt = json.load(f)

    tracklets = list(gt.keys())
    random.shuffle(tracklets)
    n_val = int(len(tracklets) * val_ratio)
    val_ids = set(tracklets[:n_val])

    val_gt = {}
    new_train_gt = {}
    for tid, label in gt.items():
        src = train_img_dir / tid
        if tid in val_ids:
            val_gt[tid] = label
            if src.exists():
                shutil.move(str(src), str(val_img_dir / tid))
        else:
            new_train_gt[tid] = label

    with open(val_gt_path, "w") as f:
        json.dump(val_gt, f)
    with open(train_gt_path, "w") as f:
        json.dump(new_train_gt, f)

    print(f"Created val split — train: {len(new_train_gt)}, val: {len(val_gt)}")

In [ ]:
# Train the digit count classifier (10% of data, SGD, 15 epochs)
%cd /content/jersey-number-pipeline
!python3 digit_count_classifier.py --train --data ./data/SoccerNet --sample_fraction 0.1

In [ ]:
# Test the trained model (prints accuracy, precision, recall, F1)
import glob, os
model_files = sorted(glob.glob('./experiments/digit_count_resnet34_*.pth'))
if not model_files:
    print("No trained model found in ./experiments/")
else:
    latest_model = model_files[-1]
    print(f"Testing with: {latest_model}")
    print(f"File size: {os.path.getsize(latest_model) / 1024 / 1024:.1f} MB")
    !python3 digit_count_classifier.py --data ./data/SoccerNet --trained_model_path {latest_model}

In [ ]:
# Save trained model to Google Drive
import shutil, os, glob
from pathlib import Path

model_files = sorted(glob.glob('./experiments/digit_count_resnet34_*.pth'))
if not model_files:
    print("No model to save!")
else:
    latest_model = model_files[-1]
    drive_dest = Path('/content/drive/MyDrive/jersey-number-pipeline-models/')
    drive_dest.mkdir(parents=True, exist_ok=True)
    dest_path = drive_dest / os.path.basename(latest_model)
    shutil.copy(latest_model, str(dest_path))
    print(f"Model saved to: {dest_path}")
    print(f"Size: {os.path.getsize(str(dest_path)) / 1024 / 1024:.1f} MB")

    # Verify it loads correctly
    import torch
    from networks import DigitCountClassifier
    model = DigitCountClassifier()
    state = torch.load(str(dest_path), map_location='cpu', weights_only=True)
    model.load_state_dict(state)
    print("Load verification: OK")